In [ ]:
%pip install -q openai-whisper librosa soundfile scikit-learn tqdm pysentimiento


In [ ]:
import whisper
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pysentimiento import create_analyzer

modelo_whisper = whisper.load_model("tiny")
analisador_sentimento = create_analyzer(task="sentiment", lang="pt")

# Usa como fonte oficial o dataset versionado no GitHub (1.180 WAVs).
from pathlib import Path
import urllib.request
import zipfile

URL_DATASET = "https://github.com/Val-Faria/TechChallengeFase4/raw/refs/heads/main/audio/datasets/dataset_audio_emocoes_1180.zip"
ZIP_DATASET = "/content/dataset_audio_github.zip"
PASTA_DATASET = Path("/content/dataset_audio_github")
urllib.request.urlretrieve(URL_DATASET, ZIP_DATASET)
PASTA_DATASET.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_DATASET, "r") as zip_ref:
    zip_ref.extractall(PASTA_DATASET)

df_meta = pd.read_csv(PASTA_DATASET / "metadata.csv")
df_meta["caminho"] = df_meta["arquivo"].map(lambda nome: str(PASTA_DATASET / nome))
quantidade_wav = len(list(PASTA_DATASET.rglob("*.wav")))
if quantidade_wav != len(df_meta):
    raise ValueError(f"Contagem divergente: {quantidade_wav} WAVs e {len(df_meta)} linhas de metadados.")
print("WAVs no dataset correto:", quantidade_wav)
print(df_meta["classe"].value_counts())

resultados = []

for _, linha in tqdm(
    df_meta.iterrows(),
    total=len(df_meta)
):
    arquivo = linha["caminho"]

    audio, sr = librosa.load(
        arquivo,
        sr=16000,
        mono=True
    )

    transcricao = modelo_whisper.transcribe(
        arquivo,
        language="pt",
        fp16=False
    )

    texto = transcricao["text"].strip()
    quantidade_palavras = len(texto.split())

    duracao = librosa.get_duration(
        y=audio,
        sr=sr
    )

    rms = librosa.feature.rms(y=audio)[0]
    energia_media = float(np.mean(rms))

    intervalos = librosa.effects.split(
        audio,
        top_db=30
    )

    tempo_sonoro = sum(
        (fim - inicio) / sr
        for inicio, fim in intervalos
    )

    tempo_pausas = max(duracao - tempo_sonoro, 0)
    proporcao_pausas = tempo_pausas / duracao

    ritmo = (
        quantidade_palavras / (duracao / 60)
        if duracao > 0 else 0
    )

    f0, _, _ = librosa.pyin(
        audio,
        fmin=70,
        fmax=500,
        sr=sr
    )

    f0_valido = f0[~np.isnan(f0)]

    tom_medio = (
        float(np.mean(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    tom_desvio = (
        float(np.std(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    sentimento = analisador_sentimento.predict(texto) if texto else None

    resultados.append({
        "arquivo": linha["arquivo"],
        "classe": linha["classe"],
        "grupo_original": linha["arquivo_original"],
        "transcricao_whisper": texto,
        "sentimento_texto": sentimento.output if sentimento else "sem_texto",
        "confianca_sentimento": float(max(sentimento.probas.values())) if sentimento else 0.0,
        "duracao": duracao,
        "energia_media": energia_media,
        "proporcao_pausas": proporcao_pausas,
        "ritmo_palavras_minuto": ritmo,
        "tom_medio_hz": tom_medio,
        "tom_desvio_hz": tom_desvio
    })

df_audio = pd.DataFrame(resultados)

df_audio.to_csv(
    "/content/caracteristicas_audio.csv",
    index=False,
    encoding="utf-8"
)

df_audio.head()


In [ ]:
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from google.colab import drive

colunas = [
    "duracao",
    "energia_media",
    "proporcao_pausas",
    "ritmo_palavras_minuto",
    "tom_medio_hz",
    "tom_desvio_hz"
]

X = df_audio[colunas].fillna(0)
y = df_audio["classe"]
grupos = df_audio["grupo_original"]

divisor = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indice_treino, indice_teste = next(divisor.split(X, y, groups=grupos))

modelo_avaliacao = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
modelo_avaliacao.fit(X.iloc[indice_treino], y.iloc[indice_treino])
previsoes = modelo_avaliacao.predict(X.iloc[indice_teste])
print(classification_report(y.iloc[indice_teste], previsoes))

# Treina o modelo final com todos os dados após a avaliação.
modelo_audio = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
modelo_audio.fit(X, y)

drive.mount("/content/drive")
CAMINHO_MODELO = "/content/drive/MyDrive/modelo_audio_1180.joblib"
joblib.dump({"modelo": modelo_audio, "colunas": colunas}, CAMINHO_MODELO)
print(f"Modelo salvo em: {CAMINHO_MODELO}")


In [ ]:
def _extrair_caracteristicas_audio(caminho_audio):
    caminho_audio = str(caminho_audio)
    audio, sr = librosa.load(caminho_audio, sr=16000, mono=True)
    duracao = librosa.get_duration(y=audio, sr=sr)
    energia_media = float(np.mean(librosa.feature.rms(y=audio)[0]))
    intervalos = librosa.effects.split(audio, top_db=30)
    tempo_sonoro = sum((fim - inicio) / sr for inicio, fim in intervalos)
    proporcao_pausas = max(duracao - tempo_sonoro, 0) / duracao if duracao > 0 else 0

    texto = modelo_whisper.transcribe(
        caminho_audio, language="pt", fp16=False
    )["text"].strip()
    ritmo = len(texto.split()) / (duracao / 60) if duracao > 0 else 0

    f0, _, _ = librosa.pyin(audio, fmin=70, fmax=500, sr=sr)
    f0_valido = f0[~np.isnan(f0)]
    tom_medio = float(np.mean(f0_valido)) if len(f0_valido) else 0.0
    tom_desvio = float(np.std(f0_valido)) if len(f0_valido) else 0.0

    amostra = pd.DataFrame([{
        "duracao": duracao,
        "energia_media": energia_media,
        "proporcao_pausas": proporcao_pausas,
        "ritmo_palavras_minuto": ritmo,
        "tom_medio_hz": tom_medio,
        "tom_desvio_hz": tom_desvio
    }], columns=colunas).fillna(0)
    return amostra, texto


def classificar_audio(caminho_audio):
    amostra, _ = _extrair_caracteristicas_audio(caminho_audio)
    probabilidades = modelo_audio.predict_proba(amostra)[0]
    indice = int(np.argmax(probabilidades))
    return str(modelo_audio.classes_[indice]), float(probabilidades[indice])


def analisar_audio_completo(caminho_audio):
    amostra, texto = _extrair_caracteristicas_audio(caminho_audio)
    probabilidades = modelo_audio.predict_proba(amostra)[0]
    indice = int(np.argmax(probabilidades))
    sentimento = analisador_sentimento.predict(texto) if texto else None
    return {
        "classe_audio": str(modelo_audio.classes_[indice]),
        "confianca_audio": float(probabilidades[indice]),
        "transcricao": texto,
        "sentimento_texto": sentimento.output if sentimento else "sem_texto",
        "confianca_sentimento": float(max(sentimento.probas.values())) if sentimento else 0.0
    }


In [ ]:
from pathlib import Path

PASTA_AUDIOS = Path("/content/dataset_audio_github")
arquivo_teste = next(PASTA_AUDIOS.rglob("ansiedade_hesitacao_004_original.wav"))
classe, confianca = classificar_audio(str(arquivo_teste))

print(arquivo_teste.name, classe, confianca)


In [ ]:
from pathlib import Path
import pandas as pd
import soundfile as sf

PASTA_AUDIOS = Path("/content/dataset_audio_github")
arquivos_audio = sorted(PASTA_AUDIOS.rglob("*.wav"))

if not arquivos_audio:
    raise FileNotFoundError(f"Nenhum arquivo WAV encontrado em {PASTA_AUDIOS}")

print(f"WAVs encontrados: {len(arquivos_audio)}")
validos = []
invalidos = []
for arquivo in arquivos_audio:
    try:
        info = sf.info(str(arquivo))
        if info.frames > 0 and info.samplerate > 0:
            validos.append(arquivo)
        else:
            invalidos.append((arquivo, "sem frames de áudio"))
    except Exception as erro:
        invalidos.append((arquivo, str(erro)))

print(f"WAVs válidos: {len(validos)} | inválidos: {len(invalidos)}")
for arquivo, motivo in invalidos:
    try:
        arquivo.resolve().relative_to(Path("/content").resolve())
        arquivo.unlink()
        print(f"Removido da cópia temporária: {arquivo.name} ({motivo})")
    except (ValueError, OSError) as erro:
        print(f"Não foi possível remover {arquivo.name}: {erro}")

# Usa características e transcrições já calculadas na célula anterior;
# assim, não repete o Whisper para os 1.180 arquivos.
registros_por_arquivo = df_audio.set_index("arquivo")
resultados = []
erros_classificacao = []
for i, arquivo in enumerate(validos, start=1):
    nome = arquivo.name
    try:
        registro = registros_por_arquivo.loc[nome]
        amostra = pd.DataFrame([registro[colunas].to_dict()], columns=colunas).fillna(0)
        probabilidades = modelo_audio.predict_proba(amostra)[0]
        indice = int(np.argmax(probabilidades))
        resultados.append({
            "arquivo": str(arquivo.relative_to(PASTA_AUDIOS)),
            "classe_audio": str(modelo_audio.classes_[indice]),
            "confianca_audio": float(probabilidades[indice]),
            "transcricao": registro["transcricao_whisper"],
            "sentimento_texto": registro["sentimento_texto"],
            "confianca_sentimento": float(registro["confianca_sentimento"])
        })
    except Exception as erro:
        erros_classificacao.append({"arquivo": nome, "erro": str(erro)})
    if i % 25 == 0:
        print(f"Processados: {i}/{len(validos)}")

CAMINHO_RESULTADOS = "/content/drive/MyDrive/resultado_audio_1180.csv"
CAMINHO_ERROS = "/content/drive/MyDrive/erros_classificacao_audio_1180.csv"
pd.DataFrame(resultados).to_csv(CAMINHO_RESULTADOS, index=False, encoding="utf-8")
pd.DataFrame(erros_classificacao).to_csv(CAMINHO_ERROS, index=False, encoding="utf-8")
print(f"\nÁudios classificados: {len(resultados)}")
print(f"Falhas durante classificação: {len(erros_classificacao)}")
print(f"Resultados salvos em: {CAMINHO_RESULTADOS}")
print(f"Relatório de falhas salvo em: {CAMINHO_ERROS}")
df_resultados = pd.DataFrame(resultados)
df_resultados.head()

In [ ]:
from datetime import datetime, timezone


def processar_evento_audio(caminho_audio):
    """Processa um novo áudio e devolve um evento para a integração.

    A função é invocada quando o sistema recebe um arquivo. Ela não captura
    áudio continuamente; a captura/entrega do arquivo deve vir da aplicação.
    """
    analise = analisar_audio_completo(caminho_audio)
    return {
        "timestamp_processamento_utc": datetime.now(timezone.utc).isoformat(),
        "arquivo": Path(caminho_audio).name,
        **analise
    }

# Demonstração de evento sob demanda para um único áudio:
evento_teste = processar_evento_audio(arquivo_teste)
evento_teste